# Sin²-Multiplied Dihedral Force — Demo

This notebook demonstrates `align_angle.SinSqDihedral`, a **singularity-free**
dihedral potential:

$$U = \frac{k}{2}\bigl(1 + d\,\cos(n\phi - \phi_0)\bigr)\,\sin^2\!\theta_1\;\sin^2\!\theta_2$$

where $\phi$ is the dihedral angle and $\theta_1$, $\theta_2$ are the two bond
angles at the central atoms of the quartet $(a, b, c, d)$.

The $\sin^2\theta$ prefactors smoothly send both the potential and all forces to
zero when any three consecutive atoms become collinear, eliminating the
$1/\sin^2\theta$ singularity in the standard `hoomd.md.dihedral.Periodic`
formulation.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `k` | float | — | spring constant $[\mathrm{energy}]$ |
| `d` | float | — | sign factor ($+1$ or $-1$) |
| `n` | int | — | multiplicity |
| `phi0` | float | 0.0 | phase offset $\phi_0$ (radians) |

### Outline

1. **Force landscape** — 2-D heatmaps of the force magnitude vs. $(\theta_1, \theta_2)$
   computed by the plugin, compared with `Periodic` (which diverges).
2. **Comb polymer simulation** — A 500-backbone + 500-sidechain comb polymer with
   T-shaped junctions where collinear geometries are common.
3. **Stability comparison** — Run the same system with increasing time step `dt`
   using both `SinSqDihedral` and `Periodic`, showing that `SinSqDihedral`
   survives much larger `dt`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import hoomd
from hoomd import align_angle

print("HOOMD version:", hoomd.version.version)
device = hoomd.device.auto_select()
print("Device:", device)

## 1. Force Landscape: SinSqDihedral vs. Periodic

We scan the two bond angles $\theta_1 \in [5°, 175°]$ and $\theta_2 \in [5°, 175°]$
at a fixed dihedral angle $\phi = 60°$, and measure the **maximum force magnitude**
on any of the four atoms, using actual HOOMD force evaluations (not analytic formulas).

- **SinSqDihedral** should produce smooth, bounded forces everywhere.
- **Periodic** should diverge ($\propto 1/\sin^2\theta$) near collinear geometries
  ($\theta \to 0°$ or $180°$).

In [ ]:
def general_dihedral(theta1_deg, theta2_deg, phi_deg, d=1.0):
    """Return 4 particle positions with given bond angles and dihedral angle.

    b at origin, c along +x at distance d.
    a in the xy plane making angle θ₁ at vertex b.
    d placed so that angle at c is θ₂ and dihedral angle is φ.
    """
    t1, t2, phi = np.radians(theta1_deg), np.radians(theta2_deg), np.radians(phi_deg)
    rb = np.array([0.0, 0.0, 0.0])
    rc = np.array([d, 0.0, 0.0])
    ra = rb + d * np.array([-np.cos(np.pi - t1), np.sin(np.pi - t1), 0.0])
    rd = rc + d * np.array([np.cos(np.pi - t2),
                            -np.sin(np.pi - t2) * np.cos(phi),
                             np.sin(np.pi - t2) * np.sin(phi)])
    return [ra.tolist(), rb.tolist(), rc.tolist(), rd.tolist()]


def measure_max_force(force_obj, positions, params, L=20.0):
    """Build a 4-particle sim with given dihedral force, return max |F|.

    Returns max force magnitude, or np.inf if the simulation produces
    non-finite forces.
    """
    snap = hoomd.Snapshot(device.communicator)
    if snap.communicator.rank == 0:
        snap.configuration.box = [L, L, L, 0, 0, 0]
        snap.particles.N = 4
        snap.particles.types = ["A"]
        snap.particles.position[:] = positions
        snap.dihedrals.N = 1
        snap.dihedrals.types = ["test"]
        snap.dihedrals.typeid[0] = 0
        snap.dihedrals.group[0] = (0, 1, 2, 3)

    sim = hoomd.Simulation(device=device)
    sim.create_state_from_snapshot(snap)
    integrator = hoomd.md.Integrator(dt=0.0)
    force_obj.params["test"] = params
    integrator.forces.append(force_obj)
    sim.operations.integrator = integrator
    sim.run(0)

    forces = np.array([force_obj.forces[i] for i in range(4)])
    if not np.all(np.isfinite(forces)):
        return np.inf
    return np.max(np.linalg.norm(forces, axis=1))


# --- Scan grid ---
angles = np.linspace(5, 175, 35)  # 35×35 grid
phi_fixed = 60.0
params_dih = dict(k=5.0, d=1, n=1, phi0=0)

max_force_sinsq = np.zeros((len(angles), len(angles)))
max_force_periodic = np.zeros((len(angles), len(angles)))

print(f"Scanning {len(angles)}×{len(angles)} = {len(angles)**2} geometries ...")
for i, t1 in enumerate(angles):
    for j, t2 in enumerate(angles):
        pos = general_dihedral(t1, t2, phi_fixed)

        # SinSqDihedral
        sinsq = align_angle.SinSqDihedral()
        max_force_sinsq[i, j] = measure_max_force(sinsq, pos, params_dih)

        # Periodic
        periodic = hoomd.md.dihedral.Periodic()
        try:
            max_force_periodic[i, j] = measure_max_force(periodic, pos, params_dih)
        except Exception:
            max_force_periodic[i, j] = np.inf
    if (i + 1) % 10 == 0:
        print(f"  row {i+1}/{len(angles)} done")

print("Scan complete.")
print(f"  SinSqDihedral  max|F| range: [{max_force_sinsq.min():.4f}, {max_force_sinsq.max():.4f}]")
pf_finite = max_force_periodic[np.isfinite(max_force_periodic)]
print(f"  Periodic       max|F| range: [{pf_finite.min():.4f}, {pf_finite.max():.2f}]")
print(f"  Periodic       non-finite entries: {np.sum(~np.isfinite(max_force_periodic))}")

In [ ]:
# --- 2-D heatmaps: side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

# Shared color scale based on the SinSqDihedral range
vmax_shared = max(max_force_sinsq.max(), 20.0)

# Left: SinSqDihedral
ax = axes[0]
im0 = ax.imshow(
    max_force_sinsq,
    origin="lower",
    extent=[angles[0], angles[-1], angles[0], angles[-1]],
    aspect="equal",
    cmap="viridis",
    norm=mcolors.LogNorm(vmin=0.01, vmax=vmax_shared),
)
ax.set_xlabel(r"$\theta_2$ (deg)")
ax.set_ylabel(r"$\theta_1$ (deg)")
ax.set_title(r"SinSqDihedral — $\max|\mathbf{F}|$")
plt.colorbar(im0, ax=ax, label="max |F|", shrink=0.85)

# Right: Periodic (clip to same scale, mark divergent regions)
ax = axes[1]
periodic_clipped = np.clip(max_force_periodic, 0.01, 1e6)
im1 = ax.imshow(
    periodic_clipped,
    origin="lower",
    extent=[angles[0], angles[-1], angles[0], angles[-1]],
    aspect="equal",
    cmap="viridis",
    norm=mcolors.LogNorm(vmin=0.01, vmax=1e6),
)
ax.set_xlabel(r"$\theta_2$ (deg)")
ax.set_title(r"Periodic — $\max|\mathbf{F}|$  (diverges near collinear)")
plt.colorbar(im1, ax=ax, label="max |F|", shrink=0.85)

fig.suptitle(
    rf"Force magnitude vs. bond angles  ($\phi = {phi_fixed}°$,  k={params_dih['k']})",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

print("Left panel: SinSqDihedral forces stay bounded everywhere.")
print("Right panel: Periodic forces blow up near θ → 0° and θ → 180° (collinear).")

### Cross-section: max force vs. $\theta_1$ at $\theta_2 = 90°$

A 1-D slice through the heatmap at $\theta_2 = 90°$ clearly shows the
divergence of `Periodic` near collinearity while `SinSqDihedral` stays bounded.

In [ ]:
# 1-D cross-section at θ₂ ≈ 90°
idx_90 = np.argmin(np.abs(angles - 90))

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(angles, max_force_sinsq[:, idx_90], "o-", ms=4, lw=1.5,
            label="SinSqDihedral", color="C0")
ax.semilogy(angles, np.clip(max_force_periodic[:, idx_90], 1e-3, None),
            "s-", ms=4, lw=1.5, label="Periodic", color="C3")
ax.set_xlabel(r"$\theta_1$ (deg)")
ax.set_ylabel(r"$\max|\mathbf{F}|$  (log scale)")
ax.set_title(rf"Force cross-section at $\theta_2 = {angles[idx_90]:.0f}°$,  $\phi = {phi_fixed}°$")
ax.axvspan(0, 15, alpha=0.1, color="red", label="near-collinear zone")
ax.axvspan(165, 180, alpha=0.1, color="red")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(angles[0], angles[-1])
plt.tight_layout()
plt.show()

## 2. Comb Polymer Simulation

We build a **comb polymer** with T-shaped junctions:

- **500 backbone** beads (type `B`), linearly connected
- **500 side-chain** beads (type `S`), one branching off each backbone bead at 90°

### Topology

| Type | Groups | Force |
|------|--------|-------|
| **Backbone bonds** | $(B_i, B_{i+1})$ | `Harmonic(k=30, r0=1)` |
| **Sidechain bonds** | $(B_i, S_i)$ | `Harmonic(k=30, r0=1)` |
| **Repulsion** | all pairs | `DPDConservative(A=25, r_cut=1)` |
| **Backbone bending** | $(B_{i-1}, B_i, B_{i+1})$ | `Harmonic(k=10, t0=π)` |
| **Junction angles** | $(B_{i\pm1}, B_i, S_i)$ | `Harmonic(k=20, t0=π/2)` |
| **Junction dihedrals** | $(S_{i-1}, B_{i-1}, B_i, S_i)$ | `SinSqDihedral(k=5, d=−1, n=1)` |

We use DPD conservative repulsion (soft, bounded force $F = A(1 - r/r_\mathrm{cut})$)
instead of WCA, so that the pair force is never the stability bottleneck.
The `d=−1` choice puts the dihedral minimum at $\phi=0$ (cis), so consecutive
side-chains prefer the **same** side of the backbone.

In [ ]:
# ── Comb polymer parameters ──
N_backbone = 500
N_total = 2 * N_backbone  # one sidechain bead per backbone bead

# Particle indexing:
#   backbone beads: 0 .. N_backbone-1
#   sidechain beads: N_backbone .. 2*N_backbone-1
# Sidechain bead i is bonded to backbone bead i.

# ── Build snapshot ──
# Backbone as a gentle helix: smooth geometry, ~165° bending angles
bond_len = 1.0
helix_radius = 5.0   # large radius → gentle curvature
helix_pitch = 3.0    # rise per full turn (along z)
# Angular step per bead so that consecutive bonds ≈ bond_len
dtheta = bond_len / np.sqrt(helix_radius**2 + (helix_pitch / (2 * np.pi))**2)

bb_pos = np.zeros((N_backbone, 3))
for i in range(N_backbone):
    angle = i * dtheta
    bb_pos[i, 0] = helix_radius * np.cos(angle)
    bb_pos[i, 1] = helix_radius * np.sin(angle)
    bb_pos[i, 2] = helix_pitch * angle / (2 * np.pi)
bb_pos -= bb_pos.mean(axis=0)

# Sidechain beads: placed radially outward from the helix axis
sc_pos = np.zeros((N_backbone, 3))
for i in range(N_backbone):
    angle = i * dtheta
    radial = np.array([np.cos(angle), np.sin(angle), 0.0])
    sc_pos[i] = bb_pos[i] + bond_len * radial

# Check bond-angle distribution
if N_backbone >= 3:
    test_angles = []
    for i in range(1, min(50, N_backbone - 1)):
        v1 = bb_pos[i - 1] - bb_pos[i]
        v2 = bb_pos[i + 1] - bb_pos[i]
        cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        test_angles.append(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))
    print(f"Initial backbone bond angles: {np.mean(test_angles):.1f}° ± {np.std(test_angles):.1f}°")

# Box: must contain all particles with margin
all_pos_init = np.vstack([bb_pos, sc_pos])
extent = np.ptp(all_pos_init, axis=0).max()
L_box = extent + 20.0

snap = hoomd.Snapshot(device.communicator)
if snap.communicator.rank == 0:
    snap.configuration.box = [L_box, L_box, L_box, 0, 0, 0]
    snap.particles.N = N_total
    snap.particles.types = ["B", "S"]
    snap.particles.mass[:] = 1.0
    snap.particles.position[:N_backbone] = bb_pos
    snap.particles.position[N_backbone:] = sc_pos
    snap.particles.typeid[:N_backbone] = 0  # B
    snap.particles.typeid[N_backbone:] = 1  # S

    # ── Bonds ──
    bb_bonds = np.column_stack([np.arange(N_backbone - 1), np.arange(1, N_backbone)])
    sc_bonds = np.column_stack([np.arange(N_backbone), np.arange(N_backbone, N_total)])
    all_bonds = np.vstack([bb_bonds, sc_bonds])
    n_bonds = len(all_bonds)

    snap.bonds.N = n_bonds
    snap.bonds.types = ["backbone", "sidechain"]
    snap.bonds.group[:] = all_bonds
    snap.bonds.typeid[:len(bb_bonds)] = 0
    snap.bonds.typeid[len(bb_bonds):] = 1

    # ── Angles ──
    # Backbone bending: (i-1, i, i+1)
    bend_angles = np.column_stack([
        np.arange(N_backbone - 2),
        np.arange(1, N_backbone - 1),
        np.arange(2, N_backbone),
    ])
    # Junction angles: (B_{i-1}, B_i, S_i) and (B_{i+1}, B_i, S_i) for internal beads
    junc_angles_left = np.column_stack([
        np.arange(N_backbone - 2),
        np.arange(1, N_backbone - 1),
        np.arange(N_backbone + 1, N_total - 1),
    ])
    junc_angles_right = np.column_stack([
        np.arange(2, N_backbone),
        np.arange(1, N_backbone - 1),
        np.arange(N_backbone + 1, N_total - 1),
    ])
    all_angles = np.vstack([bend_angles, junc_angles_left, junc_angles_right])
    n_angles = len(all_angles)

    snap.angles.N = n_angles
    snap.angles.types = ["bend", "junction"]
    snap.angles.group[:] = all_angles
    snap.angles.typeid[:len(bend_angles)] = 0
    snap.angles.typeid[len(bend_angles):] = 1

    # ── Dihedrals ──
    # Junction dihedrals only: (S_{i-1}, B_{i-1}, B_i, S_i) for i = 1..N-1
    junc_dihedrals = np.column_stack([
        np.arange(N_backbone, N_backbone + N_backbone - 1),  # S_{i-1}
        np.arange(N_backbone - 1),                            # B_{i-1}
        np.arange(1, N_backbone),                             # B_i
        np.arange(N_backbone + 1, N_total),                   # S_i
    ])
    n_dihedrals = len(junc_dihedrals)

    snap.dihedrals.N = n_dihedrals
    snap.dihedrals.types = ["junction"]
    snap.dihedrals.group[:] = junc_dihedrals
    snap.dihedrals.typeid[:] = 0

print(f"Comb polymer: {N_backbone} backbone + {N_backbone} sidechain = {N_total} particles")
print(f"  {n_bonds} bonds ({len(bb_bonds)} backbone + {len(sc_bonds)} sidechain)")
print(f"  {n_angles} angles ({len(bend_angles)} bend + {len(junc_angles_left) + len(junc_angles_right)} junction)")
print(f"  {n_dihedrals} junction dihedrals")
print(f"  Box L = {L_box:.1f}")

In [ ]:
# ── Set up simulation ──
sim = hoomd.Simulation(device=device, seed=42)
sim.create_state_from_snapshot(snap)

# Harmonic bonds
bond_force = hoomd.md.bond.Harmonic()
bond_force.params["backbone"] = dict(k=30.0, r0=1.0)
bond_force.params["sidechain"] = dict(k=30.0, r0=1.0)

# DPD Conservative repulsion (soft, bounded — no r⁻¹³ divergence)
nlist = hoomd.md.nlist.Cell(buffer=0.4, exclusions=["bond", "1-3"])
dpdc = hoomd.md.pair.DPDConservative(nlist=nlist, default_r_cut=1.0)
dpdc.params[("B", "B")] = dict(A=25.0)
dpdc.params[("B", "S")] = dict(A=25.0)
dpdc.params[("S", "S")] = dict(A=25.0)

# Harmonic angles
angle_force = hoomd.md.angle.Harmonic()
angle_force.params["bend"] = dict(k=10.0, t0=np.pi)          # straight backbone
angle_force.params["junction"] = dict(k=20.0, t0=np.pi / 2)  # T-junction (90°)

# SinSqDihedral (our plugin!)
# d=-1 → minimum at φ=0 (cis): consecutive side-chains on the same side
sinsq_force = align_angle.SinSqDihedral()
sinsq_force.params["junction"] = dict(k=5.0, d=-1, n=1, phi0=0)

# Langevin thermostat
kT = 1.0
dt_comb = 0.005
langevin = hoomd.md.methods.Langevin(filter=hoomd.filter.All(), kT=kT)
integrator = hoomd.md.Integrator(
    dt=dt_comb,
    methods=[langevin],
    forces=[bond_force, dpdc, angle_force, sinsq_force],
)
sim.operations.integrator = integrator

print(f"Forces: {[f.__class__.__name__ for f in integrator.forces]}")
print(f"dt = {dt_comb}, kT = {kT}")

In [ ]:
# ── Equilibrate & production run ──
thermo = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
sim.operations.computes.append(thermo)

# Warmup with tiny dt to resolve any initial stress
print("Warmup with dt=0.0001 (5k steps) ...", end=" ", flush=True)
integrator.dt = 0.0001
sim.run(5_000)
print(f"done. KE/N = {thermo.kinetic_energy / N_total:.3f}")

print("Warmup with dt=0.001 (5k steps) ...", end=" ", flush=True)
integrator.dt = 0.001
sim.run(5_000)
print(f"done. KE/N = {thermo.kinetic_energy / N_total:.3f}")

# Ramp to production dt
integrator.dt = dt_comb
print(f"Equilibrating at dt={dt_comb} (20k steps) ...", end=" ", flush=True)
sim.run(20_000)
print(f"done. KE/N = {thermo.kinetic_energy / N_total:.3f}")

# Production
n_prod = 50_000
log_every = 500
timesteps_log = []
pe_log = []
ke_log = []
max_force_log = []

print(f"Production run ({n_prod} steps) ...")
for step in range(n_prod // log_every):
    sim.run(log_every)
    timesteps_log.append(sim.timestep)
    pe_log.append(thermo.potential_energy)
    ke_log.append(thermo.kinetic_energy)

    # Measure max force from the dihedral force only
    f_dih = np.array([sinsq_force.forces[i] for i in range(N_total)])
    max_force_log.append(np.max(np.linalg.norm(f_dih, axis=1)))

    if (step + 1) % 20 == 0:
        print(f"  step {sim.timestep:>7d}  PE = {pe_log[-1]:>10.1f}  "
              f"KE/N = {ke_log[-1]/N_total:.3f}  max|F_dih| = {max_force_log[-1]:.2f}")

timesteps_log = np.array(timesteps_log)
pe_log = np.array(pe_log)
ke_log = np.array(ke_log)
max_force_log = np.array(max_force_log)

print(f"\nSimulation complete: {sim.timestep} steps")

In [ ]:
# ── Plot energy & max dihedral force ──
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

time = timesteps_log * dt_comb

ax = axes[0]
ax.plot(time, pe_log, lw=0.8, color="C0")
ax.set_ylabel("Potential energy")
ax.set_title(f"Comb polymer (N={N_total}) with SinSqDihedral")
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(time, ke_log / N_total, lw=0.8, color="C1")
ax.axhline(1.5 * kT, ls="--", color="gray", lw=0.8, label=f"3kT/2 = {1.5*kT}")
ax.set_ylabel("KE / N")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(time, max_force_log, lw=0.8, color="C2")
ax.set_ylabel(r"max $|\mathbf{F}_{\mathrm{dih}}|$")
ax.set_xlabel("Time")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean PE = {pe_log[len(pe_log)//2:].mean():.1f}")
print(f"Mean KE/N = {ke_log[len(ke_log)//2:].mean()/N_total:.4f}  (target: {1.5*kT})")
print(f"Max dihedral force (production): {max_force_log.max():.2f}")

### 3D visualization

The backbone is drawn as a blue line, side-chain bonds as short red sticks.
Every 10th side-chain is drawn to keep the plot readable.

In [ ]:
# Read final positions sorted by tag
with sim.state.cpu_local_snapshot as ss:
    tags = np.array(ss.particles.tag[:])
    pos = np.array(ss.particles.position[:])
order = np.argsort(tags)
pos = pos[order]

bb_pos = pos[:N_backbone]
sc_pos = pos[N_backbone:]

fig = go.Figure()

# Backbone (blue line)
fig.add_trace(go.Scatter3d(
    x=bb_pos[:, 0], y=bb_pos[:, 1], z=bb_pos[:, 2],
    mode="lines",
    line=dict(color=np.arange(N_backbone), colorscale="Viridis", width=4),
    name="Backbone",
    hoverinfo="skip",
))

# Side-chain sticks (all particles, red lines from backbone to sidechain)
sc_x, sc_y, sc_z = [], [], []
for i in range(N_backbone):
    sc_x.extend([bb_pos[i, 0], sc_pos[i, 0], None])
    sc_y.extend([bb_pos[i, 1], sc_pos[i, 1], None])
    sc_z.extend([bb_pos[i, 2], sc_pos[i, 2], None])

fig.add_trace(go.Scatter3d(
    x=sc_x, y=sc_y, z=sc_z,
    mode="lines",
    line=dict(color="red", width=2),
    name="Side-chains",
    hoverinfo="skip",
))

# Sidechain bead markers (all)
fig.add_trace(go.Scatter3d(
    x=sc_pos[:, 0],
    y=sc_pos[:, 1],
    z=sc_pos[:, 2],
    mode="markers",
    marker=dict(size=2, color="red"),
    name="Sidechain beads",
    hoverinfo="skip",
    showlegend=False,
))

fig.update_layout(
    title=f"Comb polymer — {N_backbone} backbone + {N_backbone} sidechain beads",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
    ),
    width=900, height=650,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 3. Stability Comparison: `SinSqDihedral` vs. `Periodic`

We stress-test both dihedral formulations on the same comb polymer by:
1. **Perturbing** the equilibrated positions by $\pm 0.5\,\sigma$ (random uniform kicks),
   creating a far-from-equilibrium starting point;
2. **Weakening the junction angle restraint** ($k_\mathrm{angle} = 2$, vs. $20$
   in Section 2);
3. **Stiffening the dihedral** ($k_\mathrm{dih} = 100$, vs. $5$).

The combination of a bad initial state and weak angle restraints forces many
junction angles through near-collinear configurations where the `Periodic`
dihedral diverges as $1/\sin\theta$.

For each `dt` we launch a fresh simulation from the perturbed snapshot
and run 10 000 steps. A run "survives" if no particle escapes the box and
kinetic energy stays finite.

In [ ]:
def build_comb_sim(device, snap, dihedral_force, dt, k_junc_angle=2.0, seed=123):
    """Build a comb polymer simulation with the given dihedral force.

    All other forces (bonds, DPD conservative, angles) are the same as Section 2,
    except k_junc_angle which defaults to a weak 2.0 for stress-testing.
    Returns (sim, thermo_compute).
    """
    sim_test = hoomd.Simulation(device=device, seed=seed)
    sim_test.create_state_from_snapshot(snap)

    bf = hoomd.md.bond.Harmonic()
    bf.params["backbone"] = dict(k=30.0, r0=1.0)
    bf.params["sidechain"] = dict(k=30.0, r0=1.0)

    nl = hoomd.md.nlist.Cell(buffer=0.4, exclusions=["bond", "1-3"])
    dpdc_f = hoomd.md.pair.DPDConservative(nlist=nl, default_r_cut=1.0)
    dpdc_f.params[("B", "B")] = dict(A=25.0)
    dpdc_f.params[("B", "S")] = dict(A=25.0)
    dpdc_f.params[("S", "S")] = dict(A=25.0)

    af = hoomd.md.angle.Harmonic()
    af.params["bend"] = dict(k=10.0, t0=np.pi)
    af.params["junction"] = dict(k=k_junc_angle, t0=np.pi / 2)

    lang = hoomd.md.methods.Langevin(filter=hoomd.filter.All(), kT=1.0)
    integ = hoomd.md.Integrator(
        dt=dt, methods=[lang],
        forces=[bf, dpdc_f, af, dihedral_force],
    )
    sim_test.operations.integrator = integ

    tc = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
    sim_test.operations.computes.append(tc)

    return sim_test, tc


def test_stability(device, snap, dihedral_force, dt, n_steps=10_000, check_every=1000):
    """Run a simulation for n_steps and check if it survives.

    Returns dict with keys:
        survived (bool), steps_completed (int), max_ke (float)
    """
    sim_test, tc = build_comb_sim(device, snap, dihedral_force, dt)

    steps_done = 0
    max_ke = 0.0
    survived = True

    try:
        for _ in range(n_steps // check_every):
            sim_test.run(check_every)
            steps_done += check_every

            # Check for NaN positions
            s = sim_test.state.get_snapshot()
            if s.communicator.rank == 0:
                pos_now = np.array(s.particles.position)
                if not np.all(np.isfinite(pos_now)):
                    survived = False
                    break

            ke = tc.kinetic_energy
            if not np.isfinite(ke) or ke / N_total > 1e6:
                survived = False
                break
            max_ke = max(max_ke, ke)

    except Exception as e:
        survived = False
        print(f"    Exception at step {steps_done}: {e}")

    del sim_test
    return dict(survived=survived, steps_completed=steps_done, max_ke=max_ke)


# ── Perturb the equilibrated snapshot ──
equil_snap = sim.state.get_snapshot()


def perturb_snap(snap, amplitude, rng_seed=999):
    """Return a copy of *snap* with random position displacements.

    Each coordinate is shifted by Uniform(-amplitude, +amplitude).
    This creates a far-from-equilibrium starting point where many
    dihedral inner angles pass through 0 or pi (collinear), probing
    exactly the regime where the 1/sin(theta) singularity matters.
    """
    s = hoomd.Snapshot(snap.communicator)
    if s.communicator.rank == 0:
        s.configuration.box = snap.configuration.box
        N = snap.particles.N
        s.particles.N = N
        s.particles.types = snap.particles.types
        s.particles.mass[:] = np.array(snap.particles.mass)
        s.particles.typeid[:] = np.array(snap.particles.typeid)
        rng = np.random.default_rng(rng_seed)
        pos = np.array(snap.particles.position)
        pos += rng.uniform(-amplitude, amplitude, pos.shape)
        s.particles.position[:] = pos
        # Copy topology
        s.bonds.N = snap.bonds.N
        s.bonds.types = snap.bonds.types
        s.bonds.group[:] = np.array(snap.bonds.group)
        s.bonds.typeid[:] = np.array(snap.bonds.typeid)
        s.angles.N = snap.angles.N
        s.angles.types = snap.angles.types
        s.angles.group[:] = np.array(snap.angles.group)
        s.angles.typeid[:] = np.array(snap.angles.typeid)
        s.dihedrals.N = snap.dihedrals.N
        s.dihedrals.types = snap.dihedrals.types
        s.dihedrals.group[:] = np.array(snap.dihedrals.group)
        s.dihedrals.typeid[:] = np.array(snap.dihedrals.typeid)
    return s


perturb_amplitude = 0.5  # ±0.5 σ random kick — substantial vs bond length 1
stress_snap = perturb_snap(equil_snap, perturb_amplitude)
print(f"Perturbed equilibrated positions by ±{perturb_amplitude} (bond length = 1).")
print("Stress-test settings: junction angle k ↓ 2 (weak), dihedral k ↑ 100 (stiff)")
print("  → starts far from equilibrium; junction angles explore collinear geometries\n")

# ── dt values to test ──
dt_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]
k_dih_test = 100.0  # stiffer than Section 2 to accentuate the difference

results_sinsq = {}
results_periodic = {}

print(f"Testing stability with {len(dt_values)} dt values × 2 force types ...")
print(f"  Using {N_total}-particle comb polymer, 10k steps each\n")

for dt_val in dt_values:
    print(f"dt = {dt_val}:")

    # SinSqDihedral (d=-1 → cis, same as Section 2)
    dih_sinsq = align_angle.SinSqDihedral()
    dih_sinsq.params["junction"] = dict(k=k_dih_test, d=-1, n=1, phi0=0)
    r = test_stability(device, stress_snap, dih_sinsq, dt_val)
    results_sinsq[dt_val] = r
    status = "✓ survived" if r["survived"] else f"✗ crashed at step {r['steps_completed']}"
    print(f"  SinSqDihedral: {status}  (max KE = {r['max_ke']:.0f})")

    # Periodic (d=-1 → cis, same equilibrium)
    dih_periodic = hoomd.md.dihedral.Periodic()
    dih_periodic.params["junction"] = dict(k=k_dih_test, d=-1, n=1, phi0=0)
    r = test_stability(device, stress_snap, dih_periodic, dt_val)
    results_periodic[dt_val] = r
    status = "✓ survived" if r["survived"] else f"✗ crashed at step {r['steps_completed']}"
    print(f"  Periodic:      {status}  (max KE = {r['max_ke']:.0f})")
    print()

In [ ]:
# ── Summary table ──
print(f"{'dt':>8s}  {'SinSqDihedral':>20s}  {'Periodic':>20s}")
print("-" * 55)
for dt_val in dt_values:
    rs = results_sinsq[dt_val]
    rp = results_periodic[dt_val]
    s_status = f"✓ {rs['steps_completed']}steps" if rs["survived"] else f"✗ @{rs['steps_completed']}"
    p_status = f"✓ {rp['steps_completed']}steps" if rp["survived"] else f"✗ @{rp['steps_completed']}"
    print(f"{dt_val:>8.3f}  {s_status:>20s}  {p_status:>20s}")

# Max stable dt
max_dt_sinsq = max((dt for dt in dt_values if results_sinsq[dt]["survived"]), default=0)
max_dt_periodic = max((dt for dt in dt_values if results_periodic[dt]["survived"]), default=0)
print(f"\nMax stable dt:  SinSqDihedral = {max_dt_sinsq},  Periodic = {max_dt_periodic}")

In [ ]:
# ── Visualization: steps completed for each dt ──
fig, ax = plt.subplots(figsize=(9, 4.5))

x = np.arange(len(dt_values))
width = 0.35

steps_sinsq = [results_sinsq[dt]["steps_completed"] for dt in dt_values]
steps_periodic = [results_periodic[dt]["steps_completed"] for dt in dt_values]

bars1 = ax.bar(x - width / 2, steps_sinsq, width, label="SinSqDihedral", color="C0")
bars2 = ax.bar(x + width / 2, steps_periodic, width, label="Periodic", color="C3", alpha=0.8)

# Mark crashes with ✗
for i, dt_val in enumerate(dt_values):
    if not results_sinsq[dt_val]["survived"]:
        ax.text(i - width / 2, steps_sinsq[i] + 200, "✗", ha="center", fontsize=12, color="C0")
    if not results_periodic[dt_val]["survived"]:
        ax.text(i + width / 2, steps_periodic[i] + 200, "✗", ha="center", fontsize=12, color="C3")

ax.set_xticks(x)
ax.set_xticklabels([f"{dt:.3f}" for dt in dt_values])
ax.set_xlabel("Time step dt")
ax.set_ylabel("Steps completed (out of 10 000)")
ax.set_title("Simulation stability: SinSqDihedral vs. Periodic\n(comb polymer, ✗ = crashed)")
ax.legend()
ax.set_ylim(0, 11500)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Summary

1. **Force landscape:** The `SinSqDihedral` force magnitude is smooth and bounded
   across all bond-angle combinations. The standard `Periodic` dihedral diverges
   as $\theta \to 0°$ or $180°$ (collinear geometries).

2. **Comb polymer:** The T-shaped comb polymer runs stably with `SinSqDihedral`
   at `dt = 0.005`, maintaining thermal equilibrium. The junction angles frequently
   approach collinearity during dynamics — exactly where the singularity-free
   formulation matters.

3. **Stability:** `SinSqDihedral` survives substantially larger time steps than
   `Periodic` on the same system. The `Periodic` dihedral crashes when thermal
   fluctuations push bond angles toward collinearity, producing runaway forces.
   `SinSqDihedral` avoids this by construction: $V \to 0$ smoothly as
   $\sin^2\theta \to 0$.